In [1]:
#####
#####This file extracts all commits of a project code AZURE ######
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from azure.devops.connection import Connection
from msrest.authentication import BasicAuthentication
from azure.devops.v7_0.git.models import GitQueryCommitsCriteria
import pandas as pd
import re 
import json
from config import AZURE_REPO_NAME, AZURE_TOKEN, AZURE_ORG, AZURE_PROJECT, PREFIX_TOOL, NAME_FILE_COMMITS

# Authentication Azure DevOps
credentials = BasicAuthentication('', AZURE_TOKEN)
connection = Connection(base_url=f"https://dev.azure.com/{AZURE_ORG}", creds=credentials)

# get client Git
git_client = connection.clients.get_git_client()

# URL
url = f"https://dev.azure.com/{AZURE_ORG}/{AZURE_PROJECT}/_apis/git/repositories/{AZURE_REPO_NAME}/refs?filter=tags/&api-version=7.1-preview.1"

data_folder = os.path.join(os.getcwd(), '..', 'data', 'extractions')
os.makedirs(data_folder, exist_ok=True)

# get commits criteria
search_criteria = GitQueryCommitsCriteria()
commits = git_client.get_commits(
    repository_id=AZURE_REPO_NAME,
    search_criteria=search_criteria,
    project=AZURE_PROJECT
)

# get all commits
search_criteria = GitQueryCommitsCriteria()
skip = 0
top = 100
all_commits = []

while True:
    commits = git_client.get_commits(
        repository_id=AZURE_REPO_NAME,
        search_criteria=search_criteria,
        project=AZURE_PROJECT,
        skip=skip,
        top=top
    )
    
    if not commits:
        break
    
    all_commits.extend(commits)
    skip += top

# Process commits
commit_data = []
for commit in all_commits:    
    commit_info = {
        'commit_id': commit.commit_id,
        'comment': commit.comment,
        'author': commit.author.name,
        'date': commit.author.date,
        'description': commit.comment,
        'pull_request_number': 0,
        'tool_ticket_commit': 0,
        'num_files_added': 0,
        'num_files_deleted': 0,
        'num_files_edited': 0
    }

    # search number of pull request in the comment
    if "pull request" in commit.comment.lower() or "merged pr" in commit.comment.lower() or "merge pull request" in commit.comment.lower():    
        match = re.search(r'(?:merged pr|merge pull request) ?(\d{6})', commit.comment, re.IGNORECASE)
        if match:
            commit_info['pull_request_number'] = int(match.group(1))

    # search number of tool ticket in the comment
    if PREFIX_TOOL in commit.comment.lower():  
        match = re.search(r'{PREFIX_TOOL}[-\s]?(\d{3,4})', commit.comment, re.IGNORECASE)
        if match:
            commit_info['tool_ticket_commit'] = int(match.group(1))

    # get changes from commit
    changes = git_client.get_changes(
        commit_id=commit.commit_id,
        repository_id=AZURE_REPO_NAME,
        project=AZURE_PROJECT
    )
    
    modified_files = []
    for change in changes.changes:
        item = change.get('item', {})
        path = item.get('path')
        change_type = change.get('changeType')
        git_object_type = item.get('gitObjectType')

        if path and git_object_type == 'blob':
            modified_files.append({
                'path': path,
                'change_type': change_type
            }) 
            change_type = change.get('changeType')
            # increase counter for each type of change
            if change_type == 'add':
                commit_info['num_files_added'] += 1
            elif change_type == 'delete':
                commit_info['num_files_deleted'] += 1
            elif change_type == 'edit':
                commit_info['num_files_edited'] += 1    

            
    commit_info['total_files'] = len(modified_files)
    commit_info['modified_files'] = json.dumps(modified_files)

    commit_data.append(commit_info)

    
# save data in CSV

commit_df = pd.DataFrame(commit_data)

commit_df.to_csv(os.path.join(data_folder, NAME_FILE_COMMITS), sep='|', index=True, index_label='id')

print(f"Data extracted and saved successfully in '{NAME_FILE_COMMITS}'")


Data extracted and saved successfully in 'azure_data.csv'
